In [3]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [5]:
#
# load final X and y arrays from 'final_X_and_y.npz'
#

from pathlib import Path
import pickle

directory_path_input = Path("D:/ECG/input_for_model") # chose the right path beforhand
loaded_arrays = np.load(directory_path_input/'X_y_input.npz', allow_pickle=True)

# chose the file to restore:
X_train = loaded_arrays['a']
y_train = loaded_arrays['b']
X_val = loaded_arrays['c']
y_val = loaded_arrays['d']
X_test = loaded_arrays['e']
y_test = loaded_arrays['f']
y_test
loaded_arrays.close()

In [7]:
# ============================================================
# 0) Configuration
# ============================================================

#SEED = 42
#SEED = 31
SEED = 7

tf.keras.utils.set_random_seed(SEED)

# Optional:
# This requests deterministic TensorFlow operations where supported.
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


OUTPUT_DIR = Path("cnn_binary_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#choose the right input_shape value for the input data used

model_config = {
    "input_shape": (1000, 1), # please adjust to current ECG sequnece length
    "blocks": 4,
    "base_filters": 16,
    "kernel_size": 20,
    "dilation_mode": "exp",
    "dropout": 0.0,
    "head_units": 32,
    "use_batchnorm": True,
    "pooling_head": "gap",
    "kernel_initializer": "he_normal",
    "learning_rate": 1e-4,
    "batch_size": 64,
    "max_epochs": 100,
    "early_stopping_patience": 10,
    "random_seed": SEED
}


# Replace this with your own class weights.
# Set class_weight = None if you do not want class weighting.
class_weight = {
    0: 0.953,
    1: 1.052
}


In [137]:
"""
# ============================================================
# 0a) Temporarily remove the feature dimension
#    (N, 200, 1) -> (N, 200)
# ============================================================

X_train_2d = X_train[:, :, 0]
X_val_2d   = X_val[:, :, 0]
X_test_2d  = X_test[:, :, 0]

print(X_train_2d.shape)
# (14378, 200)


# ============================================================
# 3) Min-Max scaling
# ============================================================

scaler = MinMaxScaler()

# IMPORTANT:
# fit only on training data
X_train_scaled = scaler.fit_transform(X_train_2d)

# Validation and test data use the SAME scaler
X_val_scaled = scaler.transform(X_val_2d)
X_test_scaled = scaler.transform(X_test_2d)


# ============================================================
# 4) Restore GRU input shape
#    (N, 200) -> (N, 200, 1)
# ============================================================

X_train = X_train_scaled[..., np.newaxis].astype(np.float32)
X_val   = X_val_scaled[..., np.newaxis].astype(np.float32)
X_test  = X_test_scaled[..., np.newaxis].astype(np.float32)
"""

(14378, 1000)


In [9]:

# ============================================================
# 1) Prepare labels
# ============================================================

def prepare_binary_labels(y):
    """
    Convert binary labels to a one-dimensional array containing 0 and 1.

    Supported inputs:
    - shape (N,)
    - shape (N, 1)
    - one-hot encoded shape (N, 2)
    """
    y = np.asarray(y)

    if y.ndim == 2 and y.shape[1] == 2:
        y = np.argmax(y, axis=1)

    elif y.ndim == 2 and y.shape[1] == 1:
        y = y.ravel()

    elif y.ndim != 1:
        raise ValueError(
            "Binary labels must have shape (N,), (N, 1), or (N, 2). "
            f"Received shape: {y.shape}"
        )

    y = y.astype(np.int32)

    unique_labels = np.unique(y)

    if not np.all(np.isin(unique_labels, [0, 1])):
        raise ValueError(
            "Binary labels must contain only 0 and 1. "
            f"Found labels: {unique_labels}"
        )

    return y


y_train_bin = prepare_binary_labels(y_train)
y_val_bin = prepare_binary_labels(y_val)
y_test_bin = prepare_binary_labels(y_test)

X_train_model = np.asarray(X_train, dtype=np.float32)
X_val_model = np.asarray(X_val, dtype=np.float32)
X_test_model = np.asarray(X_test, dtype=np.float32)

# Add a channel dimension if data have shape (N, 1000).
if X_train_model.ndim == 2:
    X_train_model = X_train_model[..., np.newaxis]

if X_val_model.ndim == 2:
    X_val_model = X_val_model[..., np.newaxis]

if X_test_model.ndim == 2:
    X_test_model = X_test_model[..., np.newaxis]
    
if X_train_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_train has model input shape {X_train_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )

if X_val_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_val has model input shape {X_val_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )


if X_test_model.shape[1:] != model_config["input_shape"]:
    raise ValueError(
        f"X_test has model input shape {X_test_model.shape[1:]}, "
        f"but expected {model_config['input_shape']}."
    )

if len(X_train_model) != len(y_train_bin):
    raise ValueError(
        f"Training data mismatch: "
        f"{len(X_train_model)} samples but {len(y_train_bin)} labels."
    )

if len(X_val_model) != len(y_val_bin):
    raise ValueError(
        f"Validation data mismatch: "
        f"{len(X_val_model)} samples but {len(y_val_bin)} labels."
    )

if len(X_test_model) != len(y_test_bin):
    raise ValueError(
        f"Test data mismatch: "
        f"{len(X_test_model)} samples but {len(y_test_bin)} labels."
    )

In [11]:

# ============================================================
# 3) CNN model builder
# ============================================================

def build_binary_cnn(
    input_shape=(1000, 1),
    blocks=4,
    base_filters=32,
    kernel_size=20,
    dilation_mode="exp",
    dropout=0.0,
    head_units=32,
    use_batchnorm=True,
    pooling_head="gap",
    kernel_initializer="he_normal"
):
    inputs = tf.keras.Input(
        shape=input_shape,
        name="ecg_input"
    )

    x = inputs

    for block_index in range(blocks):

        # Filter schedule:
        # block 1: 32 filters
        # block 2: 64 filters
        # block 3: 128 filters
        # block 4: 128 filters
        filters = base_filters * (2 ** min(block_index, 2))

        if dilation_mode == "exp":
            dilation_rate = 2 ** block_index
        elif dilation_mode == "none":
            dilation_rate = 1
        else:
            raise ValueError(
                "dilation_mode must be either 'none' or 'exp'."
            )

        x = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            dilation_rate=dilation_rate,
            padding="same",
            kernel_initializer=kernel_initializer,
            use_bias=not use_batchnorm,
            name=f"conv_block_{block_index + 1}"
        )(x)

        if use_batchnorm:
            x = tf.keras.layers.BatchNormalization(
                name=f"batchnorm_block_{block_index + 1}"
            )(x)

        x = tf.keras.layers.Activation(
            "relu",
            name=f"relu_block_{block_index + 1}"
        )(x)

        x = tf.keras.layers.MaxPooling1D(
            pool_size=2,
            name=f"maxpool_block_{block_index + 1}"
        )(x)

        if dropout > 0:
            x = tf.keras.layers.Dropout(
                rate=dropout,
                name=f"dropout_block_{block_index + 1}"
            )(x)

    if pooling_head == "gap":
        x = tf.keras.layers.GlobalAveragePooling1D(
            name="global_average_pooling"
        )(x)

    elif pooling_head == "flatten":
        x = tf.keras.layers.Flatten(
            name="flatten"
        )(x)

    else:
        raise ValueError(
            "pooling_head must be either 'gap' or 'flatten'."
        )

    if head_units > 0:
        x = tf.keras.layers.Dense(
            units=head_units,
            activation="relu",
            kernel_initializer=kernel_initializer,
            name="dense_head"
        )(x)

        if dropout > 0:
            x = tf.keras.layers.Dropout(
                rate=dropout,
                name="dense_head_dropout"
            )(x)

    outputs = tf.keras.layers.Dense(
        units=1,
        activation="sigmoid",
        name="binary_output"
    )(x)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="binary_ecg_cnn"
    )

    return model


model = build_binary_cnn(
    input_shape=model_config["input_shape"],
    blocks=model_config["blocks"],
    base_filters=model_config["base_filters"],
    kernel_size=model_config["kernel_size"],
    dilation_mode=model_config["dilation_mode"],
    dropout=model_config["dropout"],
    head_units=model_config["head_units"],
    use_batchnorm=model_config["use_batchnorm"],
    pooling_head=model_config["pooling_head"],
    kernel_initializer=model_config["kernel_initializer"]
)


In [13]:
# ============================================================
# 3) Compile model
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=model_config["learning_rate"]
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy",
            threshold=0.5
        )
    ]
)


# ============================================================
# 4) Store architecture and model summary
# ============================================================

with open(
    OUTPUT_DIR / "model_configuration.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        model_config,
        file,
        indent=4,
        default=str
    )


with open(
    OUTPUT_DIR / "model_summary.txt",
    "w",
    encoding="utf-8"
) as file:
    model.summary(
        print_fn=lambda line: file.write(line + "\n")
    )


# Store the complete Keras model architecture as JSON.
with open(
    OUTPUT_DIR / "model_architecture.json",
    "w",
    encoding="utf-8"
) as file:
    file.write(model.to_json())



In [15]:
# ============================================================
# 5) Callbacks
# ============================================================

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=model_config["early_stopping_patience"],
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=OUTPUT_DIR / "best_binary_cnn.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        filename=OUTPUT_DIR / "training_log.csv",
        separator=",",
        append=False
    ),

    tf.keras.callbacks.TerminateOnNaN()
]


In [17]:
# ============================================================
# 6) Train model
# ============================================================

history = model.fit(
    X_train_model,
    y_train_bin,
    validation_data=(
        X_val_model,
        y_val_bin
    ),
    epochs=model_config["max_epochs"],
    batch_size=model_config["batch_size"],
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)


# ============================================================
# 7) Store training history
# ============================================================

history_df = pd.DataFrame(history.history)
history_df.index = np.arange(1, len(history_df) + 1)
history_df.index.name = "epoch"

history_df.to_csv(
    OUTPUT_DIR / "training_history.csv"
)


# ============================================================
# 8) Generate class predictions
# ============================================================

train_probabilities = model.predict(
    X_train_model,
    batch_size=model_config["batch_size"],
    verbose=0
).ravel()

val_probabilities = model.predict(
    X_val_model,
    batch_size=model_config["batch_size"],
    verbose=0
).ravel()


test_probabilities = model.predict(
    X_test_model,
    batch_size=model_config["batch_size"],
    verbose=0
).ravel()


classification_threshold = 0.5

train_predictions = (
    train_probabilities >= classification_threshold
).astype(np.int32)

val_predictions = (
    val_probabilities >= classification_threshold
).astype(np.int32)


test_predictions = (
    test_probabilities >= classification_threshold
).astype(np.int32)


# ============================================================
# 9) Calculate final metrics
# ============================================================

train_loss, train_keras_accuracy = model.evaluate(
    X_train_model,
    y_train_bin,
    batch_size=model_config["batch_size"],
    verbose=0
)

val_loss, val_keras_accuracy = model.evaluate(
    X_val_model,
    y_val_bin,
    batch_size=model_config["batch_size"],
    verbose=0
)


test_loss, test_keras_accuracy = model.evaluate(
    X_test_model,
    y_test_bin,
    batch_size=model_config["batch_size"],
    verbose=0
)


train_accuracy = accuracy_score(
    y_train_bin,
    train_predictions
)

val_accuracy = accuracy_score(
    y_val_bin,
    val_predictions
)


test_accuracy = accuracy_score(
    y_test_bin,
    test_predictions
)

train_f1 = f1_score(
    y_train_bin,
    train_predictions,
    average="binary",
    zero_division=0
)

val_f1 = f1_score(
    y_val_bin,
    val_predictions,
    average="binary",
    zero_division=0
)


test_f1 = f1_score(
    y_test_bin,
    test_predictions,
    average="binary",
    zero_division=0
)


train_confusion_matrix = confusion_matrix(
    y_train_bin,
    train_predictions,
    labels=[0, 1]
)

val_confusion_matrix = confusion_matrix(
    y_val_bin,
    val_predictions,
    labels=[0, 1]
)


test_confusion_matrix = confusion_matrix(
    y_test_bin,
    test_predictions,
    labels=[0, 1]
)


# ============================================================
# 10) Store final metrics
# ============================================================

final_metrics = {
    "classification_threshold": classification_threshold,
    "number_of_model_parameters": int(model.count_params()),

    "training": {
        "loss": float(train_loss),
        "accuracy": float(train_accuracy),
        "keras_accuracy": float(train_keras_accuracy),
        "f1_score": float(train_f1)
    },

    "validation": {
        "loss": float(val_loss),
        "accuracy": float(val_accuracy),
        "keras_accuracy": float(val_keras_accuracy),
        "f1_score": float(val_f1)
    },
#}#,

     "test": {
        "loss": float(test_loss),
        "accuracy": float(test_accuracy),
        "keras_accuracy": float(test_keras_accuracy),
        "f1_score": float(test_f1)
    }
   
}
 

with open(
    OUTPUT_DIR / "final_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_metrics,
        file,
        indent=4
    )


# Also store the main metrics in table form.

metrics_df = pd.DataFrame(
    {
        "dataset": ["training", "validation"],
        "loss": [train_loss, val_loss],
        "accuracy": [train_accuracy, val_accuracy],
        "f1_score": [train_f1, val_f1]
    }
)


metrics_df = pd.DataFrame(
    {
        "dataset": ["training", "validation", "test"],
        "loss": [train_loss, val_loss, test_loss],
        "accuracy": [train_accuracy, val_accuracy, test_accuracy],
        "f1_score": [train_f1, val_f1, test_f1]
    }
)


metrics_df.to_csv(
    OUTPUT_DIR / "final_metrics.csv",
    index=False
)


# ============================================================
# 11) Store confusion matrices
# ============================================================

confusion_labels = [
    "predicted_class_0",
    "predicted_class_1"
]

confusion_index = [
    "true_class_0",
    "true_class_1"
]


train_cm_df = pd.DataFrame(
    train_confusion_matrix,
    index=confusion_index,
    columns=confusion_labels
)

val_cm_df = pd.DataFrame(
    val_confusion_matrix,
    index=confusion_index,
    columns=confusion_labels
)


test_cm_df = pd.DataFrame(
    test_confusion_matrix,
    index=confusion_index,
    columns=confusion_labels
)


train_cm_df.to_csv(
    OUTPUT_DIR / "training_confusion_matrix.csv"
)

val_cm_df.to_csv(
    OUTPUT_DIR / "validation_confusion_matrix.csv"
)


test_cm_df.to_csv(
    OUTPUT_DIR / "test_confusion_matrix.csv"
)

# ============================================================
# 12) Store detailed classification reports
# ============================================================

train_report = classification_report(
    y_train_bin,
    train_predictions,
    labels=[0, 1],
    target_names=["class_0", "class_1"],
    output_dict=True,
    zero_division=0
)

val_report = classification_report(
    y_val_bin,
    val_predictions,
    labels=[0, 1],
    target_names=["class_0", "class_1"],
    output_dict=True,
    zero_division=0
)


test_report = classification_report(
    y_test_bin,
    test_predictions,
    labels=[0, 1],
    target_names=["class_0", "class_1"],
    output_dict=True,
    zero_division=0
)


train_report_df = pd.DataFrame(train_report).transpose()
val_report_df = pd.DataFrame(val_report).transpose()
test_report_df = pd.DataFrame(test_report).transpose()


train_report_df.to_csv(
    OUTPUT_DIR / "training_classification_report.csv"
)

val_report_df.to_csv(
    OUTPUT_DIR / "validation_classification_report.csv"
)

test_report_df.to_csv(
    OUTPUT_DIR / "test_classification_report.csv"
)




# ============================================================
# 13) Store predictions and probabilities
# ============================================================

train_prediction_df = pd.DataFrame(
    {
        "true_label": y_train_bin,
        "predicted_probability_class_1": train_probabilities,
        "predicted_label": train_predictions
    }
)

val_prediction_df = pd.DataFrame(
    {
        "true_label": y_val_bin,
        "predicted_probability_class_1": val_probabilities,
        "predicted_label": val_predictions
    }
)


test_prediction_df = pd.DataFrame(
    {
        "true_label": y_test_bin,
        "predicted_probability_class_1": test_probabilities,
        "predicted_label": test_predictions
    }
)


train_prediction_df.to_csv(
    OUTPUT_DIR / "training_predictions.csv",
    index=False
)

val_prediction_df.to_csv(
    OUTPUT_DIR / "validation_predictions.csv",
    index=False
)


test_prediction_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)


# ============================================================
# 14) Save final model
# ============================================================

model.save(
    OUTPUT_DIR / "final_binary_cnn.keras"
)


# ============================================================
# 15) Print summary
# ============================================================

print("\nTraining completed.")
print(f"Results directory: {OUTPUT_DIR.resolve()}")
print(f"Number of parameters: {model.count_params():,}")

print("\nTraining metrics:")
print(f"Loss:     {train_loss:.4f}")
print(f"Accuracy: {train_accuracy:.4f}")
print(f"F1-score: {train_f1:.4f}")
print("Confusion matrix:")
print(train_confusion_matrix)

print("\nTraining classification report:")
print(train_report_df)


print("\nValidation metrics:")
print(f"Loss:     {val_loss:.4f}")
print(f"Accuracy: {val_accuracy:.4f}")
print(f"F1-score: {val_f1:.4f}")
print("Confusion matrix:")
print(val_confusion_matrix)

print("\nValidation classification report:")
print(val_report_df)


print("\nTest metrics:")
print(f"Loss:     {test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1-score: {test_f1:.4f}")
print("Confusion matrix:")
print(test_confusion_matrix)

print("\nTest classification report:")
print(test_report_df)


Epoch 1/100
674/674 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.6247 - loss: 0.6446
Epoch 1: val_loss improved from None to 0.58343, saving model to cnn_binary_results\best_binary_cnn.keras
674/674 ━━━━━━━━━━━━━━━━━━━━ 98s 141ms/step - accuracy: 0.6709 - loss: 0.6096 - val_accuracy: 0.6863 - val_loss: 0.5834
Epoch 2/100
674/674 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.7134 - loss: 0.5640
Epoch 2: val_loss improved from 0.58343 to 0.57661, saving model to cnn_binary_results\best_binary_cnn.keras
674/674 ━━━━━━━━━━━━━━━━━━━━ 97s 144ms/step - accuracy: 0.7152 - loss: 0.5588 - val_accuracy: 0.6921 - val_loss: 0.5766
Epoch 3/100
674/674 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.7273 - loss: 0.5444
Epoch 3: val_loss improved from 0.57661 to 0.57487, saving model to cnn_binary_results\best_binary_cnn.keras
674/674 ━━━━━━━━━━━━━━━━━━━━ 137s 136ms/step - accuracy: 0.7277 - loss: 0.5425 - val_accuracy: 0.6947 - val_loss: 0.5749
Epoch 4/100
674/674 ━━━━━━━━━━━━━━━━━━━━ 0s 144m